In [ ]:
"""
King County Housing Price Prediction with Deep Neural Network (DNN)
---------------------------------------------------------------------

This notebook trains a simple Deep Neural Network (DNN) using TensorFlow/Keras
to predict house prices based on multiple features.

Covers:
- Data preprocessing and feature engineering
- Scaling features
- Building and training a DNN
- Evaluating performance
- Single house price prediction example
"""


# Basic Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn Utilities
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, explained_variance_score

# TensorFlow and Keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [ ]:

# Load dataset
df = pd.read_csv('./data/kc_house_data.csv')

# Convert 'date' to datetime
df['date'] = pd.to_datetime(df['date'])

# Extract year and month of sale
df['year_sold'] = df['date'].dt.year
df['month_sold'] = df['date'].dt.month

# Create 'year_renovated_rev' (latest renovation year or built year)
df['year_renovated_rev'] = df.apply(
    lambda row: row['yr_renovated'] if row['yr_renovated'] > 0 else row['yr_built'], axis=1
)

# Create 'is_renovated' (boolean flag for renovation)
df['is_renovated'] = df['yr_renovated'] > 0

# Drop unnecessary columns
df.drop(['id', 'date'], axis=1, inplace=True)

print(df.head())


In [ ]:

X = df.drop('price', axis=1)
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [ ]:

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:

model = Sequential([
    Dense(units=22, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dense(units=22, activation='relu'),
    Dense(units=22, activation='relu'),
    Dense(units=22, activation='relu'),
    Dense(units=1)
])

model.compile(optimizer='adam', loss='mse')


In [ ]:

history = model.fit(
    X_train_scaled, y_train,
    epochs=250,
    validation_data=(X_test_scaled, y_test)
)


In [ ]:

predictions = model.predict(X_test_scaled)

rmse = np.sqrt(mean_squared_error(y_test, predictions))
explained_var = explained_variance_score(y_test, predictions)

print(f"Root Mean Squared Error: {rmse:.2f}")
print(f"Explained Variance Score: {explained_var:.2f}")


In [ ]:

fig = plt.figure(figsize=(10, 8))
sns.scatterplot(x=y_test.values, y=predictions.flatten())
plt.plot(y_test.values, y_test.values, 'r', label='Perfect Prediction')
plt.xlabel('Actual Price')
plt.ylabel('Predicted Price')
plt.title('Actual vs Predicted House Prices')
plt.legend()
plt.show()


In [ ]:

single_house = df.drop('price', axis=1).iloc[0].values.reshape(1, -1)
single_house_scaled = scaler.transform(pd.DataFrame(single_house, columns=X.columns))
predicted_price = model.predict(single_house_scaled)[0][0]

print(f"Predicted Price for First House: ${predicted_price:,.2f}")
print(f"Actual Price for First House: ${df.loc[0, 'price']:,.2f}")
